# 01 · GzDRL의 물리 스텝과 구동기 지연

이 노트북은 [GzDRL 논문](https://arxiv.org/html/2609.13243v1)의 식 (1)·(2)와 V-F를 이해하기 위한 **교육용 1차원 고도 toy model**이다. 실제 Gazebo, ROS 2, PPO, 13차원 강체 동역학, 역진자, 실기체 실험을 재현하지 않는다. 출력 수치는 논문 측정치가 아니다. 외부 패키지는 NumPy만 사용한다.

논문의 식 (2)는 $u_{n+1}=u_n e^{-\Delta t/t_c}+u_{\mathrm{ref},n}(1-e^{-\Delta t/t_c})$이다. 식 (1)의 수직 병진 운동만 떼어내어 $\dot z=v$, $\dot v=u/m-g$로 단순화한다. 이때 자세는 항상 수평이고 힘·토크 교란은 없다고 가정한다. 물리 주파수 1 kHz와 제어 주파수 100 Hz라면 정책 동작 한 번마다 $K=10$개의 물리 스텝이 실행된다.


In [ ]:
import numpy as np

DT = 0.001  # s: 물리 스텝
K = 10      # 1 kHz / 100 Hz
G = 9.81   # m/s^2

def actuator_step(u, u_ref, dt, tau):
    if dt <= 0 or tau <= 0:
        raise ValueError('dt와 tau는 양수여야 합니다.')
    alpha = np.exp(-dt / tau)
    return alpha * u + (1.0 - alpha) * u_ref

tau = 0.025
u0, u_ref = 0.0, 12.0
iterative = np.array([u0] + [0.0] * 100, dtype=float)
for n in range(100):
    iterative[n + 1] = actuator_step(iterative[n], u_ref, DT, tau)
n = np.arange(101)
closed_form = u_ref + (u0 - u_ref) * np.exp(-n * DT / tau)
assert np.allclose(iterative, closed_form, rtol=0, atol=1e-12)
assert np.all(np.diff(iterative) > 0)
assert np.isclose(K * DT, 0.01)
print(f'K={K}, 정책 간격={K * DT:.3f} s, 100번째 구동기 출력={iterative[-1]:.4f} N')


## 고도 운동과 정책-물리 시간 척도

정책 명령은 $K$회 물리 갱신 동안 일정하게 유지하지만, 구동기 상태 $u$와 고도 상태 $(z,v)$는 매 물리 스텝 갱신한다. 여기서는 명시적 Euler 적분을 사용한다. 논문의 Gazebo 적분기와 동일하지 않다.


In [ ]:
NOMINAL_MASS = 1.2  # kg: 논문 기체 파라미터가 아닌 toy 값

def simulate_altitude(command_offsets, mass=NOMINAL_MASS, tau=0.025):
    if mass <= 0:
        raise ValueError('mass는 양수여야 합니다.')
    z = v = 0.0
    u = NOMINAL_MASS * G
    states = [(z, v, u)]
    for offset in command_offsets:
        u_ref = NOMINAL_MASS * G + float(offset)
        for _ in range(K):
            u = actuator_step(u, u_ref, DT, tau)
            v += DT * (u / mass - G)
            z += DT * v
            states.append((z, v, u))
    return np.asarray(states, dtype=float)

hover = simulate_altitude(np.zeros(100))
assert hover.shape == (100 * K + 1, 3)
assert np.allclose(hover[:, :2], 0.0, rtol=0, atol=1e-12)
assert np.allclose(hover[:, 2], NOMINAL_MASS * G, rtol=0, atol=1e-12)
commands = np.r_[np.zeros(25), np.full(25, 0.5), np.zeros(50)]
nominal = simulate_altitude(commands)
slow_actuator = simulate_altitude(commands, tau=0.050)
assert np.allclose(nominal[:25 * K + 1, :2], 0.0, rtol=0, atol=1e-12)
assert nominal[26 * K, 2] > slow_actuator[26 * K, 2]
assert np.isfinite(nominal).all()
print(f'명령 수={len(commands)}, 물리 갱신 수={len(nominal) - 1}, 최종 고도={nominal[-1, 0]:.4f} m (toy)')


## V-F 도메인 무작위화의 축소 실습

논문 V-F는 역진자-쿼드로터에서 질량 $m$, 관성 $J$, 상승·하강 구동기 시정수 $t_{\mathrm{up}},t_{\mathrm{down}}$를 공칭값의 $[0.9,1.1]$배로 독립적으로 조정하고 10에피소드마다 다시 뽑는다. 아래는 **질량과 단일 시정수만** 바꾸는 1차원 예제다. 일반화 성능이나 논문의 보상값을 재현하지 않는다.


In [ ]:
rng = np.random.default_rng(7)
mass_scale = np.repeat(rng.uniform(0.9, 1.1, size=2), 10)
tau_scale = np.repeat(rng.uniform(0.9, 1.1, size=2), 10)
final_heights = np.array([
    simulate_altitude(commands, NOMINAL_MASS * ms, 0.025 * ts)[-1, 0]
    for ms, ts in zip(mass_scale, tau_scale)
])
assert len(final_heights) == 20
assert np.all((0.9 <= mass_scale) & (mass_scale <= 1.1))
assert np.all((0.9 <= tau_scale) & (tau_scale <= 1.1))
assert np.unique(mass_scale[:10]).size == np.unique(mass_scale[10:]).size == 1
assert np.unique(tau_scale[:10]).size == np.unique(tau_scale[10:]).size == 1
assert np.std(final_heights) > 0
print('20개 toy 에피소드의 최종 고도 평균/표준편차:', np.mean(final_heights), np.std(final_heights))
